# 🧩 Blackboard Systems vs a Sequential Pipeline

A **Blackboard System** coordinates specialist agents through a shared data store (the *blackboard*). Instead of a
fixed A → B → C hand-off, a **Controller** reads the blackboard after every contribution and decides which
specialist should act next. The workflow emerges from the problem instead of being hard-coded.

This notebook puts the two designs side by side on a query with **conditional logic**: *"find the news; if it is
neutral or positive run a technical analysis, if it is negative run a financial analysis"*. A fixed pipeline
cannot skip a step; a blackboard controller can. The comparison at the end is computed in code from which
specialists actually ran.

## Learning Objectives
In this notebook, you will learn:
1. **Tool-using specialists** - Run specialists as small `create_agent` loops that actually execute their search tool
2. **Sequential baseline** - Build a fixed `news → technical → financial → writer` pipeline with `StateGraph`
3. **Blackboard state and controller routing** - Give every specialist one shared list, and let an LLM controller pick the next specialist with a typed (`Literal`) decision
4. **Running the blackboard** - Stream the run and watch each contribution land on the board
5. **Measured comparison** - Check in code whether each system respected the *either/or* instruction

## Where this fits
Modules 06–07 routed work through a supervisor or passed control between peers. A blackboard keeps a central
controller but moves all shared context into one structure every agent can see. `02_blackboard_research_team.ipynb`
builds a fuller team with fact-check loop-backs on the same idea.

## Prerequisites
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`
- `TAVILY_API_KEY` in the same `.env` (web search)

| Property | Value |
|---|---|
| Origin | HEARSAY-II speech-recognition system (Erman et al., 1980) — the canonical Blackboard architecture |


### Definition
A **Blackboard System** is a multi-agent architecture where multiple specialist agents collaborate by reading from and writing to a shared, central data repository called the 'blackboard'. A controller or scheduler dynamically determines which agent should act next based on the evolving state of the solution on the blackboard.

### High-level Workflow

1.  **Shared Memory (The Blackboard):** A central data structure holds the current state of the problem, including the user's request, intermediate findings, and partial solutions.
2.  **Specialist Agents:** A pool of independent agents, each with a specific expertise. A specialist stays idle until the Controller activates it. (In the original HEARSAY-II, the specialists' trigger conditions watched the board and a scheduler chose among them; in this notebook the Controller does all the watching.)
3.  **Controller:** A central 'controller' agent reads the blackboard after every contribution. Its job is to analyze the current state and decide which specialist agent is best equipped to make the next contribution.
4.  **Opportunistic Activation:** The Controller activates the chosen agent. The agent reads the relevant data from the blackboard, performs its task, and writes its findings back to the blackboard.
5.  **Iteration:** The process repeats, with the Controller activating different agents in a dynamic sequence, until it determines that the solution on the blackboard is complete.

### When to Use / Applications
*   **Complex, Ill-Structured Problems:** Ideal for problems where the solution path is not known in advance and requires an emergent, opportunistic strategy (e.g., complex diagnostics, scientific discovery).
*   **Multi-Modal Systems:** A great way to coordinate agents that work with different data types (text, images, code), as they can all post their findings to the shared blackboard.
*   **Dynamic Sense-Making:** Situations that require synthesizing information from many disparate, asynchronous sources.

### Strengths & Weaknesses
*   **Strengths:**
    *   **Flexibility & Adaptability:** The workflow is not hardcoded; it emerges based on the problem, making the system highly adaptive.
    *   **Modularity:** It's very easy to add or remove specialist agents without re-architecting the entire system.
*   **Weaknesses:**
    *   **Controller Complexity:** The intelligence of the entire system depends heavily on the sophistication of the Controller. A naive Controller can lead to inefficient or looping behavior.
    *   **Debugging Challenges:** The non-linear, emergent nature of the workflow can sometimes make it harder to trace and debug than a simple sequential process.

---
## 📦 Step 1: Environment Setup

The setup cell loads `OPENAI_API_KEY` and `TAVILY_API_KEY`, imports the building blocks and sets the loop guards.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

import operator
from typing import Annotated, List, Literal, Optional, TypedDict

from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_tavily import TavilySearch
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field
from rich.console import Console
from rich.markdown import Markdown

console = Console()

# Loop guards for the blackboard controller (it can cycle, the pipeline cannot)
MAX_SPECIALIST_CALLS = 6   # enforced in code by the controller: a clean stop
RECURSION_LIMIT = 20       # LangGraph's hard cap on graph steps: an error, last resort
from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY and TAVILY_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: shared by every specialist and the controller
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini", temperature=0)

print(f"✅ LLM initialized: {llm.model_name}")

---
## 🔍 Part 1: A Search-Enabled Specialist

Each analyst needs live information, so it runs as a tiny tool-calling agent: `create_agent` binds the Tavily
search tool **and executes the calls** in a loop until the model writes its answer. (Binding a tool with
`llm.bind_tools()` alone only lets the model *request* a search; nothing runs it, and the reply is often an empty
message holding the tool call.)

In [ ]:
# ============================================================================
# SEARCH-ENABLED SPECIALIST: create_agent runs the tool loop
# ============================================================================
search_tool = TavilySearch(max_results=2)


def run_specialist(instructions: str, task: str) -> str:
    """Run a small tool-calling agent and return its final answer text."""
    agent = create_agent(llm, tools=[search_tool], system_prompt=instructions)
    result = agent.invoke({"messages": [HumanMessage(content=task)]}, {"recursion_limit": 12})
    return result["messages"][-1].content


print("✅ run_specialist() ready: Tavily search + LLM in a create_agent loop")

---
## ➡️ Part 2: The Baseline - A Sequential Pipeline

The four specialists run in a fixed order. Each one receives the previous report as context, and every node
appends its name to `steps` (an `operator.add` list), so the run leaves a record of what executed.

In [ ]:
# ============================================================================
# SEQUENTIAL PIPELINE: news -> technical -> financial -> writer
# ============================================================================
class SequentialState(TypedDict):
    user_request: str
    news_report: Optional[str]
    technical_report: Optional[str]
    financial_report: Optional[str]
    final_report: Optional[str]
    steps: Annotated[List[str], operator.add]


def news_node(state: SequentialState) -> dict:
    console.print("--- (Sequential) NEWS ANALYST ---")
    report = run_specialist(
        "You are an expert News Analyst. Search for the latest major news on the topic and summarise it concisely, "
        "stating whether its overall sentiment is positive, neutral or negative.",
        state["user_request"],
    )
    return {"news_report": report, "steps": ["News Analyst"]}


def technical_node(state: SequentialState) -> dict:
    console.print("--- (Sequential) TECHNICAL ANALYST ---")
    report = run_specialist(
        "You are an expert Technical Analyst. Using the news report as context, search for and analyse the stock's "
        "recent price action and technical indicators.",
        f"News report:\n{state['news_report']}",
    )
    return {"technical_report": report, "steps": ["Technical Analyst"]}


def financial_node(state: SequentialState) -> dict:
    console.print("--- (Sequential) FINANCIAL ANALYST ---")
    report = run_specialist(
        "You are an expert Financial Analyst. Using the news report as context, search for and analyse the "
        "company's recent financial performance.",
        f"News report:\n{state['news_report']}",
    )
    return {"financial_report": report, "steps": ["Financial Analyst"]}


def writer_node(state: SequentialState) -> dict:
    console.print("--- (Sequential) REPORT WRITER ---")
    prompt = f"""Synthesize the reports below into one cohesive answer to the user's request.

User Request: {state['user_request']}

News Report: {state['news_report']}

Technical Report: {state['technical_report']}

Financial Report: {state['financial_report']}"""
    return {"final_report": llm.invoke(prompt).content, "steps": ["Report Writer"]}


seq_builder = StateGraph(SequentialState)
seq_builder.add_node("news", news_node)
seq_builder.add_node("tech", technical_node)
seq_builder.add_node("finance", financial_node)
seq_builder.add_node("writer", writer_node)
seq_builder.add_edge(START, "news")
seq_builder.add_edge("news", "tech")
seq_builder.add_edge("tech", "finance")
seq_builder.add_edge("finance", "writer")
seq_builder.add_edge("writer", END)
sequential_app = seq_builder.compile()

print("✅ Sequential pipeline compiled. Edges:")
for edge in sequential_app.get_graph().edges:
    print(f"  {edge.source} -> {edge.target}{'  (conditional)' if edge.conditional else ''}")

### Running the pipeline on a conditional request

In [ ]:
# ============================================================================
# RUN THE SEQUENTIAL PIPELINE
# ============================================================================
dynamic_query = (
    "Find the latest major news about Nvidia. Based on the sentiment of that news, conduct either a technical "
    "analysis (if the news is neutral or positive) or a financial analysis of their recent performance "
    "(if the news is negative)."
)

final_seq_output = sequential_app.invoke({"user_request": dynamic_query, "steps": []})

print("Steps executed:", " -> ".join(final_seq_output["steps"]))
console.print(Markdown(final_seq_output["final_report"]))

**Reading the output:** the `Steps executed` line always shows `News Analyst -> Technical Analyst -> Financial
Analyst -> Report Writer`. The graph has no way to skip a branch, so the pipeline runs **both** analyses whatever
the news says. The *either/or* instruction is ignored by construction.

---
## 🧠 Part 3: The Blackboard System

**What we are going to do:**
1. **Blackboard state:** a shared `blackboard` list that every specialist reads and appends to
2. **Specialists:** the same analysts, now reading the whole blackboard for context
3. **Controller:** an LLM that reads the blackboard and returns a typed `ControllerDecision`. The `Literal` type
   means it can only name a real node (or `FINISH`)
4. **Loop guard:** the controller stops in code after `MAX_SPECIALIST_CALLS`, before asking the model

> **Key Insight**: the LLM decision is written to state (`next_agent`); a separate, pure routing function reads it.

In [ ]:
# ============================================================================
# BLACKBOARD STATE, SPECIALISTS AND CONTROLLER
# ============================================================================
AGENTS = ["News Analyst", "Technical Analyst", "Financial Analyst", "Report Writer"]


class BlackboardState(TypedDict):
    user_request: str
    blackboard: List[str]                       # every contribution, in order
    next_agent: Optional[str]                   # the controller's latest decision
    steps: Annotated[List[str], operator.add]   # which specialists ran


class ControllerDecision(BaseModel):
    next_agent: Literal["News Analyst", "Technical Analyst", "Financial Analyst", "Report Writer", "FINISH"] = Field(
        description="The specialist to run next, or FINISH once the Report Writer has posted the final answer."
    )
    reasoning: str = Field(description="A brief reason grounded in the blackboard.")


def make_blackboard_specialist(agent_name: str, persona: str, uses_search: bool = True):
    instructions = (
        f"You are an expert {persona}. Read the user request and the blackboard, perform ONLY your own function, "
        f"and reply with a concise markdown report."
    )

    def node(state: BlackboardState) -> dict:
        console.print(f"--- (Blackboard) {agent_name} is working ---")
        board = "\n---\n".join(state["blackboard"]) or "(empty)"
        task = f"User Request: {state['user_request']}\n\nBlackboard:\n{board}"
        report = run_specialist(instructions, task) if uses_search else llm.invoke(instructions + "\n\n" + task).content
        return {"blackboard": state["blackboard"] + [f"**Report from {agent_name}:**\n{report}"], "steps": [agent_name]}

    return node


news_bb = make_blackboard_specialist("News Analyst", "News Analyst who states whether the news sentiment is positive, neutral or negative")
technical_bb = make_blackboard_specialist("Technical Analyst", "Technical Analyst")
financial_bb = make_blackboard_specialist("Financial Analyst", "Financial Analyst")
writer_bb = make_blackboard_specialist("Report Writer", "Report Writer who synthesizes the final answer from the blackboard", uses_search=False)

controller_llm = llm.with_structured_output(ControllerDecision)


def controller_node(state: BlackboardState) -> dict:
    if len(state["steps"]) >= MAX_SPECIALIST_CALLS:
        console.print(f"--- CONTROLLER: {MAX_SPECIALIST_CALLS} specialist calls reached -> FINISH ---")
        return {"next_agent": "FINISH"}

    board = "\n\n".join(state["blackboard"]) or "The blackboard is currently empty."
    prompt = f"""You are the controller of a multi-agent system. Decide which specialist should run next.

Original User Request:
{state['user_request']}

Current Blackboard:
---
{board}
---

Available specialists: {', '.join(AGENTS)}

1. Work out the next logical step towards a complete answer, following any conditions in the request exactly.
2. Choose the single best specialist for that step. Do not run specialists the request says to skip.
3. Choose FINISH only after the Report Writer has posted the final answer."""
    decision = controller_llm.invoke(prompt)
    console.print(f"--- CONTROLLER: -> {decision.next_agent} | {decision.reasoning} ---")
    return {"next_agent": decision.next_agent}


print("✅ Blackboard specialists and controller defined")

### Wiring the blackboard graph
The controller is the hub: it routes to the chosen specialist, and every specialist routes back to it.

In [ ]:
# ============================================================================
# BUILD THE BLACKBOARD GRAPH
# ============================================================================
def route_to_agent(state: BlackboardState) -> str:
    return state["next_agent"]


bb_builder = StateGraph(BlackboardState)
bb_builder.add_node("Controller", controller_node)
bb_builder.add_node("News Analyst", news_bb)
bb_builder.add_node("Technical Analyst", technical_bb)
bb_builder.add_node("Financial Analyst", financial_bb)
bb_builder.add_node("Report Writer", writer_bb)

bb_builder.add_edge(START, "Controller")
bb_builder.add_conditional_edges("Controller", route_to_agent, {**{a: a for a in AGENTS}, "FINISH": END})
for agent_name in AGENTS:
    bb_builder.add_edge(agent_name, "Controller")

blackboard_app = bb_builder.compile()
print("✅ Blackboard graph compiled. Edges:")
for edge in blackboard_app.get_graph().edges:
    print(f"  {edge.source} -> {edge.target}{'  (conditional)' if edge.conditional else ''}")

---
## ▶️ Part 4: Running the Blackboard on the Same Query

`stream_mode="values"` yields the full state after every step, so each new blackboard entry can be printed as it
lands (truncated here; the final report is printed in full).

In [ ]:
# ============================================================================
# RUN THE BLACKBOARD SYSTEM
# ============================================================================
initial_bb_input = {"user_request": dynamic_query, "blackboard": [], "next_agent": None, "steps": []}

final_bb_output, seen = None, 0
for state in blackboard_app.stream(initial_bb_input, {"recursion_limit": RECURSION_LIMIT}, stream_mode="values"):
    final_bb_output = state
    for entry in state["blackboard"][seen:]:
        print(f"\n📌 New blackboard entry:\n{entry[:400]}{'...' if len(entry) > 400 else ''}\n")
    seen = len(state["blackboard"])

print("Steps executed:", " -> ".join(final_bb_output["steps"]))
console.print(Markdown(final_bb_output["blackboard"][-1]))

**Reading the output:** the controller's printed reasons show it reading the News Analyst's sentiment before
choosing between the Technical and Financial Analyst. Compare the `Steps executed` line with the pipeline's.
Whether the controller picked the right branch is checked in code in Part 5 rather than taken on trust.

---
## 📏 Part 5: Measuring Instruction-Following in Code

Two checks per system, both computed from `steps`:
- **Followed either/or**: exactly one of the Technical/Financial analysts ran
- **Correct branch**: the branch that ran matches the news sentiment. The sentiment is classified once, from the
  blackboard's News Analyst report, with a narrow structured-output call (one label, no scoring)

In [ ]:
# ============================================================================
# CODE-COMPUTED COMPARISON
# ============================================================================
class Sentiment(BaseModel):
    sentiment: Literal["positive", "neutral", "negative"]


news_entry = next(e for e in final_bb_output["blackboard"] if e.startswith("**Report from News Analyst"))
sentiment = llm.with_structured_output(Sentiment).invoke(
    f"Classify the overall sentiment of this news report about the company.\n\n{news_entry}"
).sentiment
expected_branch = "Financial Analyst" if sentiment == "negative" else "Technical Analyst"
print(f"News sentiment: {sentiment} -> the request asks for the {expected_branch} only\n")


def check(steps: List[str]) -> dict:
    ran = [s for s in steps if s in ("Technical Analyst", "Financial Analyst")]
    return {
        "specialist calls": len(steps),
        "branches run": ", ".join(ran) or "none",
        "followed either/or": "✅" if len(set(ran)) == 1 else "❌",
        "correct branch": "✅" if set(ran) == {expected_branch} else "❌",
    }


rows = {"Sequential": check(final_seq_output["steps"]), "Blackboard": check(final_bb_output["steps"])}
print(f"{'':12}" + "".join(f"{k:>22}" for k in rows["Sequential"]))
for name, row in rows.items():
    print(f"{name:12}" + "".join(f"{str(v):>22}" for v in row.values()))

**Reading the table:** the sequential row always shows both branches and ❌ for *followed either/or*. That is a
structural limit, not a model error. The blackboard row shows what this run's controller did. A ❌ there is
possible (the controller is an LLM), which is exactly why the check is computed rather than assumed. If it
happens, rerun the cell or tighten the controller prompt.

---
## 📝 Summary

### 1. Two coordination styles
- **Sequential pipeline**: fixed edges, predictable and cheap, but unable to skip a step the task makes unnecessary
- **Blackboard**: shared `blackboard` list plus a controller that re-decides after every contribution

### 2. Making the blackboard safe and honest
- `ControllerDecision.next_agent` is a `Literal`, so the controller can only name a real node
- `MAX_SPECIALIST_CALLS` stops the loop in code; `recursion_limit` is the backstop
- Specialists run as `create_agent` loops, so their search tool is actually executed
- The comparison is computed from `steps`, not judged by another LLM

### Next Steps
- `02_blackboard_research_team.ipynb` builds a four-agent research team whose controller loops back when a
  fact-check fails